In [ ]:
# %% [code] {"jupyter":{"outputs_hidden":false}}
import os
import time
import glob
import random
import itertools
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim_metric

# ==========================================
# 1. DATASET & DATALOADERS
# ==========================================
class UnalignedDataset(Dataset):
    """
    Loads unpaired images from two domains (Domain A and Domain B).
     Because this is an "unpaired" image-to-image translation task (unlike Pix2Pix), 
    we randomly shuffle the datasets so that image A[i] is NOT structurally aligned with image B[i].
    """
    def __init__(self, data_dir, transform=None):
        self.transform = transform
        
        print("1. Loading Dataset...")
        all_paths = []
        for root, dirs, files in os.walk(data_dir):
            # Skip any subdirectories that contain expert masks or segmentations
            if any(forbidden in root.lower() for forbidden in ["mask", "annotation", "groundtruth", "segmentation"]):
                continue
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')):
                    if "mask" not in file.lower() and "expert" not in file.lower():
                        all_paths.append(os.path.join(root, file))
        
        #  A fixed seed ensures we don't accidentally leak Domain A images 
        # into Domain B if the script is interrupted and restarted.
        random.seed(42) 
        random.shuffle(all_paths)
        
        # 50/50 Split for Unpaired Translation
        split_idx = len(all_paths) // 2
        self.A_paths = all_paths[:split_idx]
        self.B_paths = all_paths[split_idx:]
        
        self.A_size = len(self.A_paths)
        self.B_size = len(self.B_paths)
        print(f"Found {len(all_paths)} total images. Split into A:{self.A_size} and B:{self.B_size}")

    def __getitem__(self, index):
        # Domain A loops sequentially, Domain B is picked randomly to break structural alignment
        A_path = self.A_paths[index % self.A_size]
        B_path = self.B_paths[random.randint(0, self.B_size - 1)]
        
        A_img = Image.open(A_path).convert('L')
        B_img = Image.open(B_path).convert('L')
        
        if self.transform:
            A_img = self.transform(A_img)
            B_img = self.transform(B_img)
            
        return {'A': A_img, 'B': B_img}

    def __len__(self):
        return max(self.A_size, self.B_size)

#  We resize to 400x400. PyTorch's ToTensor() automatically 
# scales pixel values from [0, 255] to [0.0, 1.0].
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((400, 400)), 
    transforms.ToTensor() 
])

dataloader = DataLoader(
    UnalignedDataset(data_dir="/kaggle/input", transform=transform),
    batch_size=1, # Strict requirement from the base paper
    shuffle=True
)


# ==========================================
# 2. ARCHITECTURES (Base Paper Specs)
# ==========================================
class ResidualBlock(nn.Module):
    """Standard ResNet Block to process deep feature maps without losing spatial resolution."""
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1), nn.Conv2d(channels, channels, kernel_size=3), nn.InstanceNorm2d(channels), nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1), nn.Conv2d(channels, channels, kernel_size=3), nn.InstanceNorm2d(channels)
        )
    def forward(self, x): return x + self.block(x)

class BasePaperGenerator(nn.Module):
    """
     This exactly mirrors the 2025 Base Paper architecture.
    A standard CycleGAN is a ResNet-9 (2 down, 9 res, 2 up). 
    The paper explicitly added 1 down and 1 up block to make it a ResNet-15 architecture.
    """
    def __init__(self, input_channels=1, output_channels=1, num_residual_blocks=9):
        super(BasePaperGenerator, self).__init__()
        
        # Initial Convolution (Layer 1)
        self.initial = nn.Sequential(nn.ReflectionPad2d(3), nn.Conv2d(input_channels, 64, kernel_size=7), nn.InstanceNorm2d(64), nn.ReLU(inplace=True))
        
        # 3 Downsampling Blocks (Paper added the 3rd block here)
        self.down1 = self._conv_block(64, 128)
        self.down2 = self._conv_block(128, 256)
        self.down3 = self._conv_block(256, 512)
        
        # 9 ResNet Bottlenecks
        self.resnet_bottleneck = nn.Sequential(*[ResidualBlock(512) for _ in range(num_residual_blocks)])
        
        # 3 Upsampling Blocks (Paper added the 1st block here)
        self.up1 = self._upconv_block(512, 256)  
        self.up2 = self._upconv_block(256, 128)   
        self.up3 = self._upconv_block(128, 64)
        
        #  Using Sigmoid instead of Tanh because our images are strictly [0, 1]
        self.final = nn.Sequential(nn.ReflectionPad2d(3), nn.Conv2d(64, output_channels, kernel_size=7), nn.Sigmoid())

    def _conv_block(self, in_c, out_c): 
        #  We use InstanceNorm2d natively in PyTorch instead of hacking GroupNorm like we do in Keras.
        return nn.Sequential(nn.Conv2d(in_c, out_c, kernel_size=3, stride=2, padding=1), nn.InstanceNorm2d(out_c), nn.ReLU(inplace=True))
        
    def _upconv_block(self, in_c, out_c): 
        return nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.ReflectionPad2d(1), nn.Conv2d(in_c, out_c, kernel_size=3, stride=1, padding=0),
            nn.InstanceNorm2d(out_c), nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # 1. Encoding (Extract early layers for NOISE/STYLE LOSS)
        c0 = self.initial(x)
        c1 = self.down1(c0)
        c2 = self.down2(c1)
        c3 = self.down3(c2)
        
        # 2. Transformation
        bot = self.resnet_bottleneck(c3)
        
        # 3. Decoding (Extract late layer for CONTENT LOSS)
        u1 = self.up1(bot)
        u2 = self.up2(u1)
        u3 = self.up3(u2) 
        
        out = self.final(u3)
        
        #  We return the image AND the intermediate feature maps required for Eq 3, 4, 5
        return out, u3, [c0, c1, c2] 

class PatchGANDiscriminator(nn.Module):
    """
    Standard 70x70 PatchGAN. 
     Instead of outputting a single True/False for the whole image, it outputs a matrix 
    where each cell judges a 70x70 patch, making it highly sensitive to local high-frequency textures.
    """
    def __init__(self, input_channels=1):
        super(PatchGANDiscriminator, self).__init__()
        def discriminator_block(in_filters, out_filters, stride=2, normalize=True):
            layers = [nn.Conv2d(in_filters, out_filters, kernel_size=4, stride=stride, padding=1)]
            if normalize: layers.append(nn.InstanceNorm2d(out_filters))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *discriminator_block(input_channels, 64, normalize=False), 
            *discriminator_block(64, 128),
            *discriminator_block(128, 256),
            *discriminator_block(256, 512, stride=1), 
            nn.Conv2d(512, 1, kernel_size=4, padding=1) 
        )
    def forward(self, x): return self.model(x)


# ==========================================
# 3. UTILITIES & CUSTOM LOSS FUNCTIONS
# ==========================================
def weights_init_normal(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        torch.nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("InstanceNorm2d") != -1:
        if hasattr(m, "weight") and m.weight is not None:
            torch.nn.init.normal_(m.weight.data, 1.0, 0.02)
            torch.nn.init.constant_(m.bias.data, 0.0)

class LambdaLR:
    """Linearly decays learning rate to 0 after the halfway point of training."""
    def __init__(self, n_epochs, decay_start_epoch):
        self.n_epochs = n_epochs
        self.decay_start_epoch = decay_start_epoch
    def step(self, epoch):
        return 1.0 - max(0, epoch - self.decay_start_epoch) / float(self.n_epochs - self.decay_start_epoch)

def calc_gram_matrix(tensor):
    """
     A Gram matrix computes the dot product of a feature map with its own transpose.
    It destroys spatial awareness but perfectly captures stylistic texture (like ultrasound speckle noise).
    """
    b, c, h, w = tensor.size()
    features = tensor.view(b, c, h * w)
    features_t = features.transpose(1, 2)
    gram = features.bmm(features_t) / (c * h * w)
    return gram.view(b, -1)

def wasserstein_1d(x, y):
    """Approximates the Wasserstein distance by sorting the distributions (Eq 5)."""
    x_sorted, _ = torch.sort(x, dim=-1)
    y_sorted, _ = torch.sort(y, dim=-1)
    return torch.mean(torch.abs(x_sorted - y_sorted))

class MultiObjectiveGeneratorLoss(nn.Module):
    """
     This implements Equation 6 from the paper exactly.
    It combines Standard Adversarial Loss, L1 Content Loss, and Wasserstein Noise Loss.
    """
    def __init__(self, lambda_content=10.0, lambda_noise=10.0):
        super(MultiObjectiveGeneratorLoss, self).__init__()
        self.lambda_content = lambda_content
        self.lambda_noise = lambda_noise
        self.adversarial_loss = nn.BCEWithLogitsLoss() 
        self.l1_loss = nn.L1Loss()           

    def forward(self, D_c_fake_out, D_n_fake_out, content_fake, content_real, noise_fake_list, noise_real_list):
        # 1. Adversarial components
        L_adv_c = self.adversarial_loss(D_c_fake_out, torch.ones_like(D_c_fake_out))
        L_adv_n = self.adversarial_loss(D_n_fake_out, torch.ones_like(D_n_fake_out))

        # 2. Content Loss (L1 distance of deep feature maps)
        L_content = self.l1_loss(content_fake, content_real)

        # 3. Noise/Style Loss (Wasserstein distance of early Gram Matrices)
        L_noise = 0
        for nf, nr in zip(noise_fake_list, noise_real_list):
            L_noise += wasserstein_1d(nf, nr)

        # Total Objective Function (Eq 6)
        L_Total = L_adv_c + L_adv_n + (self.lambda_content * L_content) + (self.lambda_noise * L_noise)
        return L_Total


# ==========================================
# 4. INITIALIZATION & TRAINING LOOP
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Instantiating Base Paper 2-Discriminator GAN on {device}...")

G = BasePaperGenerator(input_channels=1, output_channels=1).to(device)
D_c = PatchGANDiscriminator(input_channels=1).to(device) # Discriminator for Content
D_n = PatchGANDiscriminator(input_channels=1).to(device) # Discriminator for Noise/Texture

G.apply(weights_init_normal)
D_c.apply(weights_init_normal)
D_n.apply(weights_init_normal)

# Standard GAN optimizers
optimizer_G = torch.optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D_c = torch.optim.Adam(D_c.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D_n = torch.optim.Adam(D_n.parameters(), lr=0.0002, betas=(0.5, 0.999))

num_epochs = 100
lr_scheduler_G = torch.optim.lr_scheduler.LambdaLR(optimizer_G, lr_lambda=LambdaLR(num_epochs, 50).step)
lr_scheduler_D_c = torch.optim.lr_scheduler.LambdaLR(optimizer_D_c, lr_lambda=LambdaLR(num_epochs, 50).step)
lr_scheduler_D_n = torch.optim.lr_scheduler.LambdaLR(optimizer_D_n, lr_lambda=LambdaLR(num_epochs, 50).step)

criterion_multi = MultiObjectiveGeneratorLoss(lambda_content=10.0, lambda_noise=10.0).to(device)

os.makedirs("/kaggle/working/saved_images_basepaper", exist_ok=True)
os.makedirs("/kaggle/working/saved_models_basepaper", exist_ok=True)

print("Starting Base Paper GAN Training Loop (100 Epochs)...")
for epoch in range(num_epochs):
    for i, batch in enumerate(dataloader):
        real_A = batch['A'].to(device)
        real_B = batch['B'].to(device)
        
        # --- FORWARD PASSES ---
        # 1. Generate fake image, get its content features (feat_A), and get early style layers
        fake_B, feat_A, fake_B_layers = G(real_A)
        
        # 2. Extract target style features from the real Domain B image
        with torch.no_grad():
            _, _, real_B_layers = G(real_B)
            
        # 3. Convert early feature maps to Gram Matrices for Noise Loss
        fake_grams = [calc_gram_matrix(f) for f in fake_B_layers]
        real_grams = [calc_gram_matrix(r) for r in real_B_layers]

        # 4. Extract features of the newly GENERATED image for Content Loss comparison
        _, feat_fake_B, _ = G(fake_B)
            
        # --- STEP 1: UPDATE DISCRIMINATORS ---
        optimizer_D_c.zero_grad()
        optimizer_D_n.zero_grad()
        
        #  We detach fake_B here so gradients don't flow back into the Generator
        fake_B_detached = fake_B.detach()
        
        loss_D_c = (criterion_multi.adversarial_loss(D_c(real_B), torch.ones_like(D_c(real_B))) + 
                    criterion_multi.adversarial_loss(D_c(fake_B_detached), torch.zeros_like(D_c(fake_B_detached)))) / 2
        loss_D_c.backward()
        optimizer_D_c.step()
        
        loss_D_n = (criterion_multi.adversarial_loss(D_n(real_B), torch.ones_like(D_n(real_B))) + 
                    criterion_multi.adversarial_loss(D_n(fake_B_detached), torch.zeros_like(D_n(fake_B_detached)))) / 2
        loss_D_n.backward()
        optimizer_D_n.step()

        # --- STEP 2: UPDATE GENERATOR ---
        optimizer_G.zero_grad()
        
        logits_c = D_c(fake_B)
        logits_n = D_n(fake_B)
        
        loss_G_Total = criterion_multi(
            D_c_fake_out=logits_c, 
            D_n_fake_out=logits_n, 
            content_fake=feat_fake_B,   
            content_real=feat_A,        
            noise_fake_list=fake_grams, 
            noise_real_list=real_grams 
        )
        
        loss_G_Total.backward()
        optimizer_G.step()
        
    lr_scheduler_G.step()
    lr_scheduler_D_c.step()
    lr_scheduler_D_n.step()


# ==========================================
# 5. EVALUATION, METRICS, AND CLINICAL IMPACT
# ==========================================
print("\n" + "="*60)
print(" 📊 QUANTITATIVE RESULTS (COMPUTER VISION) ")
print("="*60)

#  We switch to eval() to freeze BatchNorm/InstanceNorm running stats
G.eval()

ssim_scores, hc_base, hc_adapt, bd_base, bd_adapt = [], [], [], [], []

with torch.no_grad():
    for i, batch in enumerate(dataloader):
        if i >= 100: break 
        
        real_A, real_B = batch['A'].to(device), batch['B'].to(device)
        generated, _, _ = G(real_A)
        
        img_a, img_b, img_gen = real_A[0, 0].cpu().numpy(), real_B[0, 0].cpu().numpy(), generated[0, 0].cpu().numpy()
        
        # SSIM Evaluation
        ssim_scores.append(ssim_metric(img_a, img_gen, data_range=1.0))
        
        # Histogram Evaluation (Texture Correlation)
        img_a_flat, img_b_flat, img_gen_flat = img_a.flatten(), img_b.flatten(), img_gen.flatten()
        hist_a, _ = np.histogram(img_a_flat, bins=256, range=(0, 1))
        hist_b, _ = np.histogram(img_b_flat, bins=256, range=(0, 1))
        hist_gen, _ = np.histogram(img_gen_flat, bins=256, range=(0, 1))
        
        hist_a, hist_b, hist_gen = hist_a / np.sum(hist_a), hist_b / np.sum(hist_b), hist_gen / np.sum(hist_gen)
        
        hc_base.append(np.corrcoef(hist_a, hist_b)[0, 1])
        hc_adapt.append(np.corrcoef(hist_gen, hist_b)[0, 1])
        
        bd_base.append(np.sqrt(np.maximum(0.0, 1.0 - np.sum(np.sqrt(hist_a * hist_b)))))
        bd_adapt.append(np.sqrt(np.maximum(0.0, 1.0 - np.sum(np.sqrt(hist_gen * hist_b)))))

print(f"SSIM (Whole Image):        {np.mean(ssim_scores):.2f} (±{np.std(ssim_scores):.2f})")
print(f"Histogram Correlation (HC): {np.mean(hc_adapt):.3f}")
print(f"Bhattacharyya Distance (BD): {np.mean(bd_adapt):.3f}")


print("\n" + "="*60)
print(" 📸 GENERATING VISUALIZATIONS (SSIM MAP) ")
print("="*60)
#  An SSIM Map shows pixel-by-pixel structural changes. 
# Green means perfect preservation, Red means the AI destroyed/hallucinated anatomy.

batch = next(iter(dataloader))
real_A = batch['A'].to(device)
with torch.no_grad():
    generated, _, _ = G(real_A)

img_in, img_out = real_A[0, 0].cpu().numpy(), generated[0, 0].cpu().numpy()
score, ssim_map = ssim_metric(img_in, img_out, data_range=1.0, full=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(img_in, cmap='gray'); axes[0].set_title("Input image", fontsize=16); axes[0].axis('off')
axes[1].imshow(img_out, cmap='gray'); axes[1].set_title("Generated image", fontsize=16); axes[1].axis('off')
im = axes[2].imshow(ssim_map, cmap='RdYlGn', vmin=-1.0, vmax=1.0)
axes[2].set_title(f"SSIM map (Score: {score:.3f})", fontsize=16); axes[2].axis('off')
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04).set_label('Similarity', rotation=90)
plt.show()


print("\n" + "="*60)
print(" 🏥 CLINICAL IMPACT & RISK MARKER RE-CLASSIFICATION ")
print("="*60)
#  This section calculates clinical physics (GSM) using binary masks 
# verified by experts to isolate the lumen (blood flow) and adventitia (artery wall).

DATA_DIR = "/kaggle/input"
us_dir, mask_dir = None, None
for root, dirs, files in os.walk(DATA_DIR):
    if os.path.basename(root) == "US images": us_dir = root
    elif os.path.basename(root) == "Expert mask images": mask_dir = root

if us_dir and mask_dir:
    valid_files = [f for f in os.listdir(us_dir) if f.endswith(('.png', '.jpg'))]
    gsm_before_list, gsm_after_list, reclassified = [], [], 0
    tbc_before_list, tbc_after_list = [], []
    
    def load_img_tensor(path): 
        img = Image.open(path).convert('L')
        return transform(img).unsqueeze(0).to(device)

    for filename in valid_files[:50]:
        mask_path = os.path.join(mask_dir, filename)
        if not os.path.exists(mask_path): continue

        orig_img_tensor, mask_tensor = load_img_tensor(os.path.join(us_dir, filename)), load_img_tensor(mask_path)
        with torch.no_grad():
            gen_img_tensor, _, _ = G(orig_img_tensor)
        
        orig_img, gen_img, mask = orig_img_tensor[0, 0].cpu().numpy() * 255.0, gen_img_tensor[0, 0].cpu().numpy() * 255.0, mask_tensor[0, 0].cpu().numpy()
        
        # Isolate ROI: Lumen is white mask (>0.5), Tissue is dark mask (<0.5)
        lumen_roi, tissue_roi = mask > 0.5, (mask < 0.5) & (orig_img > 10.0) 
        if np.sum(lumen_roi) < 10 or np.sum(tissue_roi) < 10: continue
            
        def calc_gsm(img): 
            #  GSM measures variance in the tissue relative to the blood.
            # When the GAN smooths out noise, it compresses this variance, altering the score!
            return np.median(190.0 * (img[tissue_roi] - np.mean(img[lumen_roi])) / max(1e-5, np.max(img[tissue_roi])))
            
        def calc_tbc(img): return 20.0 * np.log10(max(1e-5, np.mean(img[lumen_roi])) / max(1e-5, np.mean(img[tissue_roi])))

        gsm_before, gsm_after = calc_gsm(orig_img), calc_gsm(gen_img)
        tbc_before, tbc_after = calc_tbc(orig_img), calc_tbc(gen_img)
        
        gsm_before_list.append(gsm_before); gsm_after_list.append(gsm_after)
        tbc_before_list.append(tbc_before); tbc_after_list.append(tbc_after)
        
        if (gsm_before >= 25) != (gsm_after >= 25): reclassified += 1
            
    reclass_rate = (reclassified/len(gsm_before_list))*100 if len(gsm_before_list) > 0 else 0.0

    print("1. Noise Reduction (Contrast dB):")
    print(f"   * The model successfully altered the medical contrast, shifting")
    print(f"     from {np.mean(tbc_before_list):.2f} dB to {np.mean(tbc_after_list):.2f} dB.\n")
    
    print("2. Plaque Vulnerability (Grey Scale Median - GSM):")
    print(f"   * Baseline GSM: {np.mean(gsm_before_list):.1f}")
    print(f"   * Post-AI GSM:  {np.mean(gsm_after_list):.1f}")
    print(f"   * Result: The AI visually improved the images, but mathematically caused")
    print(f"     {reclass_rate:.1f}% of patients to be re-classified into the high-risk stroke category.")
print("="*60)